# JBrowse 2 in a notebook — quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/01_quickstart.ipynb)

A JBrowse 2 linear genome view rendered as an [anywidget](https://anywidget.dev), drawn on the GPU. Works in Jupyter, JupyterLab, VS Code, and Colab from a single bundle.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## An assembly and a view

An assembly is the flat `{name, uri}` shorthand — core picks the adapter from the extension and derives the index files. This reference names chromosomes `1`, `2`, … but the UCSC bigWig below uses `chr1`, `chr2`, …; `refname_aliases_uri` points at UCSC's alias table so the two line up. `location` sets the opening region.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, track

hg38 = {
    "name": "hg38",
    "uri": "https://jbrowse.org/genomes/GRCh38/fasta/hg38.prefix.fa.gz",
    "aliases": ["GRCh38"],
    "refNameAliases": {
        "uri": "https://jbrowse.org/genomes/GRCh38/hg38_aliases.txt"
    },
}

view = LinearGenomeView(assembly=hg38, location="10:29,838,565..29,838,850")
view

## Add a track

A bare data-file URL is a track — its type and adapter are inferred from the extension, the way [@jbrowse/img](https://jbrowse.org/jb2/docs/jbrowse-img)'s `--bam`/`--bigwig`/`--cram` flags work for the CLI. `track(uri, name=...)` is the same expansion with a display name and room for extra config; `assemblyNames` is filled in from the view's assembly. Here, a conservation bigwig. Any non-default setting (color, height, ...) is a key you add to the returned config dict.

In [ ]:
view.add_track(
    track(
        "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/phyloP100way/hg38.phyloP100way.bw",
        name="phyloP100way",
    )
)

## Drive the view from Python, read it back

Setting `location` navigates the view; after panning in the UI, reading `location` returns the user's current region (two-way sync).

In [ ]:
view.location = "1:1,000,000..1,050,000"

In [ ]:
view.location  # updates as you pan/zoom in the view above